## Hyperparameter search
Do this tutorial after [`run.ipynb`](./run.ipynb).

You might be wondering how we arrived at the input configurations in the 'Optimizing' and the 'Improving run time' sections in `run.ipynb`. The method [`dlkoopman.hyp_search:run_hyp_search()`](https://galoisinc.github.io/dlkoopman/hyp_search.html#dlkoopman.hyp_search.run_hyp_search) can perform hyperparameter search on either `StatePred` or `TrajPred`. This sweeps the values of the different inputs to the class and its methods. Each configuration is trained and the loss and ANAE statistics across epochs recorded for training and validation data. These results can then be used to select 'good' input configurations.

In [1]:
from dlkoopman.hyp_search import run_hyp_search
from dlkoopman.state_pred import StatePredDataHandler
from dlkoopman import utils

/Users/sourya/.venv/dlkoopman/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load data

In [2]:
import pickle
with open('./data.pkl', 'rb') as f:
    data = pickle.load(f)

The type of data handler we create determines the predictor model on which hyperparameter search is performed. In this case, we create a `StatePredDataHandler`. This automatically sets the hyperparameter search to run on `StatePred` models.

In [3]:
dh = StatePredDataHandler(
    Xtr=data['Xtr'], ttr=data['ttr'],
    Xva=data['Xva'], tva=data['tva']
)

**The data handler is the only part that has to change when switching between `TrajPred` and `StatePred`. Everything else in the hyperparameter search remains the same.**

Note that test data is not used in hyperparameter search. It is highly recommended to provide validation data though, so that validation loss and ANAE statistics can also be collected.

The following is an optional step used to seed the run. Since neural nets initialize their parameters randomly, setting the same random seed ensures reproducibility of results as long as the same `torch` version is used (we used `1.12.1`). If you do not get the exact same results as ours, don't worry, the basic aim of the tutorials is for you to understand how things work.

In [4]:
utils.set_seed(10)

### Define options for hyperparameter search
The `StatePred` class has possible inputs:
- `dh`
- `rank`
- `encoded_size`
- `encoder_hidden_layers`
- `decoder_hidden_layers`
- `batch_norm`

Its `train_net()` method has possible inputs:
- `numepochs`
- `early_stopping`
- `early_stopping_metric`
- `lr`
- `weight_decay`
- `decoder_loss_weight`
- `Kreg`
- `cond_threshold`
- `clip_grad_norm`
- `clip_grad_value`

All of these inputs, with the exception of `dh` already defined above, can be swept together in the hyperparameter search.

For more on the inputs, see the [documentation of `StatePred`](https://galoisinc.github.io/dlkoopman/state_pred.html#dlkoopman.state_pred.StatePred).

Let us define some values to sweep over.

In [5]:
hyp_options = {
    'rank': 6, # 1 option
    'encoder_hidden_layers': [ [200,200], [300,200,100] ], # 2 options
    'numepochs': [200, 300, 400], # 3 options
    'clip_grad_value': 2. # 1 option
}

Options that are not provided will revert to defaults.

### Run hyperparameter search
Let us now run the hyperparameter search.

In [6]:
try:
    run_hyp_search(
        dh = dh,
        hyp_options = hyp_options
    )
except ValueError as e:
    print(f'ValueError: {e}')

ValueError: 'encoded_size' is a required argument to StatePred, so 'hyp_options' must include a key-value pair for {'encoded_size': <encoded_size_values>}


Uh oh, you got a `ValueError: 'encoded_size' is a required argument to StatePred, so 'hyp_options' must include a key-value pair for {'encoded_size': <encoded_size_values>}`. As it says, we need to specify one or more options for `encoded_size` since this is a required argument that does not have a default value. Let us specify:

In [7]:
hyp_options['encoded_size'] = [50, 100]

`hyp_options` now has a total of 12 options = 2 for `'encoded_size'` times 2 for `'encoder_hidden_layers'` times 3 for `'numepochs'`.

Let us run it again. This may take a minute.

In [8]:
output_folder = run_hyp_search(
    dh = dh,
    hyp_options = hyp_options
)

  0%|          | 0/12 [00:00<?, ?it/s]


********************************************************************************
Starting StatePred hyperparameter search. Results will be stored in /Users/sourya/work/Essence/dlkoopman/examples/state_pred_naca0012/hyp_search_Te67h6guU84QGJzFJEkCDW/hyp_search_results.csv.

Performing total 12 runs. You can interrupt the script at any time (e.g. Ctrl+C), and intermediate results will be available in the above file.

Log of the entire hyperparameter search, as well as logs of failed StatePred runs will also be stored in the same folder.

Hyperparameters' sweep ranges:
rank : 6
encoded_size : 50, 100
encoder_hidden_layers : [200, 200], [300, 200, 100]
numepochs : 200, 300, 400
clip_grad_value : 2.0
********************************************************************************



100%|██████████| 12/12 [01:02<00:00,  5.20s/it]


### Results
Hooray, it now runs successfully and returns a path to a folder (yours will be different). This contains a file `hyp_search_results.csv`. Let us open it.

In [9]:
import pandas as pd
import os
df = pd.read_csv(os.path.join(output_folder, 'hyp_search_results.csv'))
df

,UUID,encoded_size,encoder_hidden_layers,numepochs,avg_recon_loss_tr,final_recon_loss_tr,avg_recon_loss_va,best_recon_loss_va,bestep_recon_loss_va,avg_lin_loss_tr,...,avg_lin_anae_tr,final_lin_anae_tr,avg_lin_anae_va,best_lin_anae_va,bestep_lin_anae_va,avg_pred_anae_tr,final_pred_anae_tr,avg_pred_anae_va,best_pred_anae_va,bestep_pred_anae_va
0,W9g23ZfkpHkbJNhxsDJtV8,100,"[300, 200, 100]",400,0.000663,0.000002,0.003458,0.000192,357,0.000076,...,2.869072,1.096238,10.512812,1.958855,236,50.148211,7.704675,78.809576,18.818399,350
1,B5bXaxnbFs6uMnTGNWqMbM,50,"[200, 200]",400,0.000780,0.000002,0.003783,0.000169,280,0.000023,...,2.623877,1.330029,16.078172,2.341614,41,59.023745,11.494137,95.759773,24.168331,391
2,SdKByhBGz6HTuQVswEu6SR,100,"[200, 200]",400,0.000783,0.000001,0.003534,0.000165,383,0.000055,...,5.956423,4.366830,23.197734,1.877471,36,67.412621,10.167628,108.677135,21.230761,371
3,MtjwbqV6Ug4RqLD9ADrfM3,50,"[300, 200, 100]",400,0.000725,0.000003,0.003766,0.000099,328,0.000089,...,4.273739,2.205433,11.316160,1.168934,139,58.197615,11.112122,110.512196,32.743229,368
4,e58o9rzNg3P56fMNhj6he6,50,"[300, 200, 100]",300,0.000950,0.000002,0.005151,0.000260,247,0.000052,...,3.120325,1.207940,12.752888,1.809601,80,60.941715,8.092951,115.115150,19.846489,268
5,bzzKihKhfAvUa3d4nQeDTx,100,"[300, 200, 100]",300,0.000994,0.000003,0.005172,0.000147,264,0.000161,...,4.866093,0.647565,20.026423,1.506001,201,61.982005,8.623109,126.951720,28.572554,289
6,AfUnaEVQTpek6uVoKFdw3d,100,"[200, 200]",300,0.000979,0.000002,0.004687,0.000194,255,0.000041,...,3.031048,0.798071,10.630702,2.977784,34,84.961290,10.386682,128.044908,26.545732,236
7,VdPAsgE5RpYcGS4VpqD5eA,50,"[200, 200]",300,0.000925,0.000003,0.004523,0.000228,282,0.000024,...,4.976096,1.536636,42.025757,5.117496,133,83.071532,13.694921,142.066521,40.378876,299
8,mm38khWznr2oWWAkGM59xw,100,"[300, 200, 100]",200,0.001393,0.000007,0.006817,0.000139,198,0.000174,...,6.151690,0.657052,14.414925,1.441267,51,86.163193,11.443872,150.919934,32.652920,198
9,Z6LKmwkafFsWCrBKCG2WVu,100,"[200, 200]",200,0.001660,0.000017,0.007937,0.000404,197,0.000074,...,7.000523,5.735347,40.423012,5.582700,47,117.312167,28.056973,151.411925,44.143978,197


This contains loss and ANAE statistics for all 12 runs, as `encoded_size`, `encoder_hidden_layers`, and `numepochs` are swept. The statistics collected for each performance metric `<perf>` are:
- `avg_<perf>_tr` - Average of metric for training data over all epochs.
- `final_<perf>_tr` - Value of metric in last epoch of training data.
- `avg_<perf>_va` - Average of metric for validation data over all epochs.
- `best_<perf>_va` - Best value of metric for validation data over all epochs.
- `bestep_<perf>_va` - Epoch at which best value of metric for validation data was obtained.

The 12 results from top to bottom are arranged from best to worst of the `sort_key` of `run_hyp_search()`, which is by default set to `'avg_pred_anae_va'`. This is because prediction ANAE of validation data averaged across all epochs is an important metric for quantifying performance.

Let's view this.

In [10]:
df_truncated = df[['encoded_size', 'encoder_hidden_layers', 'numepochs', 'avg_pred_anae_va']]
df_truncated

,encoded_size,encoder_hidden_layers,numepochs,avg_pred_anae_va
0,100,"[300, 200, 100]",400,78.809576
1,50,"[200, 200]",400,95.759773
2,100,"[200, 200]",400,108.677135
3,50,"[300, 200, 100]",400,110.512196
4,50,"[300, 200, 100]",300,115.115150
5,100,"[300, 200, 100]",300,126.951720
6,100,"[200, 200]",300,128.044908
7,50,"[200, 200]",300,142.066521
8,100,"[300, 200, 100]",200,150.919934
9,100,"[200, 200]",200,151.411925


### Ignoring first few epochs
The ANAE values above are really high. This is because the performance is erratic early on, before settling down. You can also see this by looking back at the results in [`run.ipynb`](./run.ipynb). Here's one example of those results.

<img src="./skewed_initial_epochs_example.png" width="300"/>

A few erratic initial epochs can skew the average statistics significantly. This is why the `run_hyp_search()` method has an argument `avg_ignore_initial_epochs`, which specifies the number of initial epochs to ignore for average calculations. Let us set this to `100`, so that averaging starts from the 100th epoch.

In [11]:
utils.set_seed(10)

output_folder = run_hyp_search(
    dh = dh,
    hyp_options = hyp_options,
    avg_ignore_initial_epochs = 100
)

df = pd.read_csv(os.path.join(output_folder, 'hyp_search_results.csv'))
df_truncated = df[['encoded_size', 'encoder_hidden_layers', 'numepochs', 'avg_pred_anae_va']]
df_truncated

  0%|          | 0/12 [00:00<?, ?it/s]


********************************************************************************
Starting StatePred hyperparameter search. Results will be stored in /Users/sourya/work/Essence/dlkoopman/examples/state_pred_naca0012/hyp_search_jUzjiQxpnQ9YQMU5FG5FQt/hyp_search_results.csv.

Performing total 12 runs. You can interrupt the script at any time (e.g. Ctrl+C), and intermediate results will be available in the above file.

Log of the entire hyperparameter search, as well as logs of failed StatePred runs will also be stored in the same folder.

Hyperparameters' sweep ranges:
rank : 6
encoded_size : 50, 100
encoder_hidden_layers : [200, 200], [300, 200, 100]
numepochs : 200, 300, 400
clip_grad_value : 2.0
********************************************************************************



100%|██████████| 12/12 [01:10<00:00,  5.86s/it]


,encoded_size,encoder_hidden_layers,numepochs,avg_pred_anae_va
0,100,"[300, 200, 100]",400,31.476379
1,50,"[300, 200, 100]",300,39.509835
2,100,"[300, 200, 100]",200,41.553454
3,50,"[200, 200]",400,41.560001
4,100,"[200, 200]",300,41.656022
5,100,"[200, 200]",400,42.146620
6,50,"[300, 200, 100]",400,46.276506
7,100,"[300, 200, 100]",300,46.455410
8,50,"[200, 200]",300,52.775295
9,100,"[200, 200]",200,64.829427


The results look a lot better now, since they are less sensitive to outlier epochs. You can see from these results that `encoder_hidden_layers = [300, 200, 100]` is generally doing better than `[200, 200]`. These insights are very helpful in selecting a good combination of hyperparameters.

### Controlling the number of runs
If you don't want to wait to run every possible configuration (which can exponentially explode as the number of options increase), you can control the number of runs using the `numruns` argument of `run_hyp_search()`. Let us set this to 5. This will randomly sample 5 runs out of the total of 12.

In [12]:
utils.set_seed(10)

output_folder = run_hyp_search(
    dh = dh,
    hyp_options = hyp_options,
    avg_ignore_initial_epochs = 100,
    numruns = 5
)

df = pd.read_csv(os.path.join(output_folder, 'hyp_search_results.csv'))
df_truncated = df[['encoded_size', 'encoder_hidden_layers', 'numepochs', 'avg_pred_anae_va']]
df_truncated

  0%|          | 0/5 [00:00<?, ?it/s]


********************************************************************************
Starting StatePred hyperparameter search. Results will be stored in /Users/sourya/work/Essence/dlkoopman/examples/state_pred_naca0012/hyp_search_nFHH2JZNPDRaEfw73VAttA/hyp_search_results.csv.

Performing total 5 runs. You can interrupt the script at any time (e.g. Ctrl+C), and intermediate results will be available in the above file.

Log of the entire hyperparameter search, as well as logs of failed StatePred runs will also be stored in the same folder.

Hyperparameters' sweep ranges:
rank : 6
encoded_size : 50, 100
encoder_hidden_layers : [200, 200], [300, 200, 100]
numepochs : 200, 300, 400
clip_grad_value : 2.0
********************************************************************************



100%|██████████| 5/5 [00:29<00:00,  5.87s/it]


,encoded_size,encoder_hidden_layers,numepochs,avg_pred_anae_va
0,100,"[300, 200, 100]",400,41.011872
1,100,"[200, 200]",400,42.566754
2,50,"[200, 200]",300,45.284574
3,50,"[300, 200, 100]",200,45.362044
4,50,"[200, 200]",200,97.725613


### Performance vs time tradeoff
We highly recommend performing hyperparameter search for any problem as it can lead to massively improved results (we got $<7\%$ prediction ANAE in [`run.ipynb`](./run.ipynb)!). The longer you perform a hyperparameter search for, the more likely you are to get good results. If required, you can perform several hundred or even several thousand runs, which can take time to run, but the results are usually worth it.

Here's an example of an extensive hyperparameter search:
```python
output_folder = run_hyp_search(
    dh = dh,
    hyp_options = {
        'rank': [3,6,8,10,20], #5 options
        'num_encoded_states': [50,100,200,500,1000], #5 options
        'encoder_hidden_layers': [
            [100,100],[200,200],[500,500],
            [50,100],[100,50],[100,200],[200,100],[200,500],[500,200],[500,1000],[1000,500],
            [100,100,100],[200,200,200],[500,500,500],
            [50,100,200],[200,100,50],[100,200,500],[500,200,100],[200,500,1000],[1000,500,200]
        ], #20 options
        'weight_decay': [0.,1e-6,1e-5,1e-4], #4 options
        'Kreg': [0.,1e-3,1e-2], #3 options
        'clip_grad_norm': [None,5.,10.], #3 options
        'clip_grad_value': [None,2.], #2 options
    }, # total = 36,000 options
    avg_ignore_initial_epochs = 100,
    numruns = 3600 # randomly sample 10% of the entire space
)
```

### A note on `decoder_loss_weight`
Since `decoder_loss_weight` is an input to `train_net()`, it is a valid key for `hyp_options`. However, note that changing `decoder_loss_weight` will change the scale of the loss function, so it won't be fair any more to compare the loss metrics across configurations with different values of `decoder_loss_weight`. The ANAE metrics can still be compared across all configurations.